# 022 — Training (unet_v2 + beta-NLL)

Trains two independent things introduced in this session, in one notebook:

- **Part A — beta-NLL retraining** of the four existing heteroscedastic architectures (`unet_nll`, `resunet_nll`, `attention_unet_nll`, `efficientnet_unet_nll`) with `scripts.losses.beta_gaussian_nll_loss` (Seitzer et al. 2022), which corrects the gradient-starvation issue of the plain Gaussian NLL used in `021_training_nll.ipynb`. Saved to a **separate checkpoint tree** (`models/nll_beta/<arch>/`), not `models/<arch>/`, so the existing `gaussian_nll` checkpoints from `021_training_nll.ipynb` are preserved — `032_evaluation_v2.ipynb` needs both to compare them.
- **Part B — `unet_v2`** (`scripts/unet_v2.py`), the standard UNet with three independently-toggleable modifications discussed in this session, all off by default and matching `unet.py` exactly when off:
  - `use_strided_conv`: learned stride-2 convolution instead of `MaxPool2D` for encoder downsampling.
  - `use_upsample_conv`: bilinear `UpSampling2D` + `Conv2D` instead of `Conv2DTranspose` for decoder upsampling (avoids checkerboard artifacts).
  - `dropout_rate`: `SpatialDropout2D` at the bottleneck and the first decoder block only, not every block (see `scripts/unet_v2.py` module docstring for why — dropout compounds with `BatchNormalization`, used in every conv block).

  Trained as a new, independent architecture name (`unet_v2`), so it never collides with the existing `unet` checkpoint — saved to `models/unet_v2/` like any other deterministic architecture.

Does not modify or retrain `unet`, `resunet`, `attention_unet`, `efficientnet_unet`, or the `gaussian_nll`-trained `*_nll` checkpoints from `021_training_nll.ipynb`.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import compile_model, get_callbacks, get_model
from scripts.trainer_nll import NLL_LOSSES, compile_model_nll, get_model_nll
from scripts.visualization import plot_training_curves

# Seed Python / NumPy / TensorFlow from settings.SEED for reproducible runs.
set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Build datasets

In [ ]:
# Mockup-aware split: real artwork groups stay leakage-free (whole group in
# one fold); the 6 mockup groups (settings.MOCKUP_ARTWORK_IDS) are split at
# the individual-pair level instead. Matches the active split in
# 020_training.ipynb / 021_training_nll.ipynb, so both parts below train on
# the same conditions as the checkpoints they will be compared against.
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Part A — beta-NLL retraining

`LOSS_NAME = "beta_nll"` (`scripts.trainer_nll.NLL_LOSSES`) weights the Gaussian NLL by `stop_gradient(sigma) ** (2 * BETA)`, counteracting the gradient starvation the plain `gaussian_nll` loss shows in high-variance regions. `BETA = 0` would recover `gaussian_nll` exactly — see `code-review.md` §7.6.

Checkpoints go to `BETA_MODEL_DIR = models/nll_beta/<arch>/`, not `models/<arch>/`, so the existing `gaussian_nll` checkpoints are untouched.

In [ ]:
ARCHS_NLL = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
EPOCHS_NLL = settings.EPOCHS  # -- lower for a quick smoke test
LOSS_NAME = "beta_nll"
BETA = 0.5

assert LOSS_NAME in NLL_LOSSES, f"LOSS_NAME must be one of {NLL_LOSSES}"

# Separate checkpoint/log tree so this run never overwrites the gaussian_nll
# checkpoints trained in 021_training_nll.ipynb.
BETA_MODEL_DIR = settings.MODELS_DIR / "nll_beta"
BETA_LOG_DIR = settings.LOGS_DIR / "nll_beta"

histories_nll: dict = {}

for arch in ARCHS_NLL:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}  (loss: {LOSS_NAME}, beta={BETA})")
    print(f"{'=' * 60}")

    model = get_model_nll(arch)
    model = compile_model_nll(
        model, lr=settings.LEARNING_RATE, loss_name=LOSS_NAME, beta=BETA
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=BETA_LOG_DIR, model_dir=BETA_MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_NLL,
        callbacks=callbacks,
        verbose=1,
    )
    histories_nll[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}, beta_nll): {best_val_loss:.4f}")

## 3. Part A — training curves

In [ ]:
for arch, history in histories_nll.items():
    fig = plot_training_curves(history, title=f"Training history — {arch} (beta_nll)")
    plt.show()

## 4. Part B — unet_v2 (architectural modifications)

Set `EPOCHS_V2 = 2` for a quick smoke test before committing to a full run. Rerunning this cell with different flag values overwrites `models/unet_v2/best_model.keras` — same caveat as `LOSS_NAME` in `021_training_nll.ipynb`.

In [ ]:
USE_STRIDED_CONV = True
USE_UPSAMPLE_CONV = True
DROPOUT_RATE = 0.2
EPOCHS_V2 = settings.EPOCHS  # -- lower for a quick smoke test

model_v2 = get_model(
    "unet_v2",
    use_strided_conv=USE_STRIDED_CONV,
    use_upsample_conv=USE_UPSAMPLE_CONV,
    dropout_rate=DROPOUT_RATE,
)
model_v2 = compile_model(
    model_v2, "unet_v2", lr=settings.LEARNING_RATE, loss_alpha=settings.LOSS_ALPHA
)
model_v2.summary(line_length=80)

callbacks_v2 = get_callbacks(
    "unet_v2", log_dir=settings.LOGS_DIR, model_dir=settings.MODELS_DIR
)

history_v2 = model_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_V2,
    callbacks=callbacks_v2,
    verbose=1,
)

best_val_loss_v2 = min(history_v2.history["val_loss"])
print(f"\nBest val_loss (unet_v2): {best_val_loss_v2:.4f}")

## 5. Part B — training curve

In [ ]:
fig = plot_training_curves(history_v2.history, title="Training history — unet_v2")
plt.show()

## 6. Summary

Checkpoints saved to:
- `models/nll_beta/<arch>/best_model.keras` for the four beta-NLL architectures (Part A) — the `gaussian_nll` checkpoints at `models/<arch>/best_model.keras` from `021_training_nll.ipynb` are untouched.
- `models/unet_v2/best_model.keras` (Part B).

Run `tensorboard --logdir logs/` to inspect curves interactively.

In [ ]:
for arch in ARCHS_NLL:
    ckpt = BETA_MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25} (beta_nll): {status}  ({ckpt})")

ckpt_v2 = settings.MODELS_DIR / "unet_v2" / "best_model.keras"
status_v2 = "found" if ckpt_v2.exists() else "MISSING"
print(f"{'unet_v2':<25}          : {status_v2}  ({ckpt_v2})")